# Autoresearch Experiment Analysis

Analysis of autonomous hyperparameter tuning results from `results.tsv`.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 5 columns: commit, val_loss, memory_gb, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df["val_loss"] = pd.to_numeric(df["val_loss"], errors="coerce")
df["memory_gb"] = pd.to_numeric(df["memory_gb"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(5)

Total experiments: 100
Columns: ['commit', 'val_loss', 'memory_gb', 'status', 'description']


,commit,val_loss,memory_gb,status,description
0,48003ad,6.630933,13.8,KEEP,baseline
1,7be3923,0.000000,0.0,CRASH,DEVICE_BATCH_SIZE 16->32 (OOM)
2,f55c53f,6.045879,9.5,KEEP,GPT-2 Small shape: 12 layers 768 embd 12 heads...
3,c7037d2,6.132749,16.7,DISCARD,DEVICE_BATCH_SIZE 16->32 with 12-layer model
4,eebc90e,6.068506,9.5,DISCARD,MAX_LR 2.4e-3 -> 3.6e-3


In [3]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

Experiment outcomes:
status
DISCARD    60
KEEP       38
CRASH       2

Keep rate: 38/98 = 38.8%


In [4]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    bpb = row["val_loss"]
    desc = row["description"]
    print(f"  #{i:3d}  bpb={bpb:.6f}  mem={row['memory_gb']:.1f}GB  {desc}")

KEPT experiments (38 total):

  #  0  bpb=6.630933  mem=13.8GB  baseline
  #  2  bpb=6.045879  mem=9.5GB  GPT-2 Small shape: 12 layers 768 embd 12 heads (was 30x512x8)
  #  9  bpb=5.894826  mem=9.5GB  WSD (warmup-stable-decay) LR schedule instead of cosine
  # 10  bpb=5.820127  mem=9.5GB  WSD decay fraction 0.2 -> 0.1
  # 16  bpb=4.922944  mem=9.5GB  TOTAL_BATCH_SIZE 524288 -> 262144 (2x optimizer steps)
  # 17  bpb=4.592337  mem=9.5GB  TOTAL_BATCH_SIZE 262144 -> 131072 (4x optimizer steps vs original)
  # 18  bpb=4.512604  mem=9.5GB  MAX_LR 2.4e-3 -> 1.8e-3 for smaller batch (sqrt scaling)
  # 19  bpb=4.138837  mem=9.5GB  TOTAL_BATCH_SIZE 65536 + MAX_LR 1.2e-3
  # 22  bpb=4.104738  mem=9.5GB  MAX_LR 1.2e-3 -> 9e-4 with batch 65536
  # 28  bpb=4.102738  mem=9.5GB  cosine decay in WSD (instead of linear)
  # 32  bpb=4.077010  mem=8.6GB  untie embeddings + n_embd 768->640 n_head 10 (65% MFU)
  # 33  bpb=4.059556  mem=9.4GB  16 layers 576 embd 9 heads untied (~122M params)
  # 34  bpb=3.9

## Val Loss Over Time

Track how the best (kept) val_loss evolves as experiments progress. The running minimum shows the "frontier" -- the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes for plotting
valid = df[df["status"] != "CRASH"].copy()
valid = valid.reset_index(drop=True)

baseline_bpb = valid.loc[0, "val_loss"]

# Only plot points at or below baseline (the interesting region)
below = valid[valid["val_loss"] <= baseline_bpb + 0.0005]

# Plot discarded as faint background dots
disc = below[below["status"] == "DISCARD"]
ax.scatter(disc.index, disc["val_loss"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Plot kept experiments as prominent green dots
kept_v = below[below["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["val_loss"],
           c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

# Running minimum step line
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_bpb = valid.loc[kept_mask, "val_loss"]
running_min = kept_bpb.cummin()
ax.step(kept_idx, running_min, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Label each kept experiment with its description
for idx, bpb in zip(kept_idx, kept_bpb):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."

    ax.annotate(desc, (idx, bpb),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Validation loss (lower is better)", fontsize=12)
ax.set_title(f"Autoresearch Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.2)

# Y-axis: from just below best to just above baseline
best = kept_bpb.min()
margin = (baseline_bpb - best) * 0.15
ax.set_ylim(best - margin, baseline_bpb + margin)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [6]:
# Summary stats
kept = df[df["status"] == "KEEP"].copy()
baseline_bpb = df.iloc[0]["val_loss"]
best_bpb = kept["val_loss"].min()
best_row = kept.loc[kept["val_loss"].idxmin()]

print(f"Baseline val_loss:  {baseline_bpb:.6f}")
print(f"Best val_loss:      {best_bpb:.6f}")
print(f"Total improvement: {baseline_bpb - best_bpb:.6f} ({(baseline_bpb - best_bpb) / baseline_bpb * 100:.2f}%)")
print(f"Best experiment:   {best_row['description']}")
print()

# How many experiments to find each improvement
print("Cumulative effort per improvement:")
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: val_loss={row['val_loss']:.6f}  {desc}")

Baseline val_loss:  6.630933
Best val_loss:      3.711456
Total improvement: 2.919477 (44.03%)
Best experiment:   AdamW beta2 0.95 -> 0.99 for embeddings

Cumulative effort per improvement:
  Experiment #  0: val_loss=6.630933  baseline
  Experiment #  2: val_loss=6.045879  GPT-2 Small shape: 12 layers 768 embd 12 heads (was 30x512x8)
  Experiment #  9: val_loss=5.894826  WSD (warmup-stable-decay) LR schedule instead of cosine
  Experiment # 10: val_loss=5.820127  WSD decay fraction 0.2 -> 0.1
  Experiment # 16: val_loss=4.922944  TOTAL_BATCH_SIZE 524288 -> 262144 (2x optimizer steps)
  Experiment # 17: val_loss=4.592337  TOTAL_BATCH_SIZE 262144 -> 131072 (4x optimizer steps vs original)
  Experiment # 18: val_loss=4.512604  MAX_LR 2.4e-3 -> 1.8e-3 for smaller batch (sqrt scaling)
  Experiment # 19: val_loss=4.138837  TOTAL_BATCH_SIZE 65536 + MAX_LR 1.2e-3
  Experiment # 22: val_loss=4.104738  MAX_LR 1.2e-3 -> 9e-4 with batch 65536
  Experiment # 28: val_loss=4.102738  cosine decay in 

## Top Hits (Kept Experiments by Improvement)

In [7]:
# Each kept experiment's delta is measured vs the previous kept experiment's bpb
# (since experiments are cumulative -- each one builds on the last kept state)
kept = df[df["status"] == "KEEP"].copy()
kept["prev_val_loss"] = kept["val_loss"].shift(1)
kept["delta"] = kept["prev_val_loss"] - kept["val_loss"]

# Drop baseline (no delta)
hits = kept.iloc[1:].copy()

# Sort by delta improvement (biggest first)
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'VAL':>10}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.6f}  {row['val_loss']:.6f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  TOTAL improvement over baseline")

Rank     Delta         VAL  Description
--------------------------------------------------------------------------------
   1  +0.897183  4.922944  TOTAL_BATCH_SIZE 524288 -> 262144 (2x optimizer steps)
   2  +0.585054  6.045879  GPT-2 Small shape: 12 layers 768 embd 12 heads (was 30x512x8)
   3  +0.373767  4.138837  TOTAL_BATCH_SIZE 65536 + MAX_LR 1.2e-3
   4  +0.330607  4.592337  TOTAL_BATCH_SIZE 262144 -> 131072 (4x optimizer steps vs original)
   5  +0.151053  5.894826  WSD (warmup-stable-decay) LR schedule instead of cosine
   6  +0.079733  4.512604  MAX_LR 2.4e-3 -> 1.8e-3 for smaller batch (sqrt scaling)
   7  +0.076739  3.982817  20 layers 512 embd 8 heads untied (~120M params)
   8  +0.074699  5.820127  WSD decay fraction 0.2 -> 0.1
   9  +0.061496  3.921321  MAX_LR 9e-4 -> 1.2e-3 for 20-layer model
  10  +0.041533  3.879788  MAX_LR 1.2e-3 -> 1.5e-3
  11  +0.034099  4.104738  MAX_LR 1.2e-3 -> 9e-4 with batch 65536
  12  +0.025728  4.077010  untie embeddings + n_embd 768->640 n